# Pipeline de Ingesta: Construcción del Grafo de Conocimiento

Sistema Graph RAG sobre el canon de Sherlock Holmes.
Este notebook ejecuta el pipeline completo de ingesta:
1. Descarga de textos de Project Gutenberg
2. Separación en relatos individuales
3. Chunking consciente de la estructura
4. Extracción multipaso de entidades y relaciones
5. Entity resolution
6. Población del grafo en Neo4j

In [1]:
%load_ext autoreload
%autoreload 2
# Si ejecutas desde notebooks/, necesitas que graphrag sea importable.
# Con uv: uv sync --extra dev && pip install -e .
from graphrag.config import get_settings
from graphrag.graph.neo4j_manager import Neo4jManager
from graphrag.ingestion.text_processor import TextProcessor
from graphrag.ingestion.entity_extractor import EntityExtractor

settings = get_settings()
print(f"Proyecto GCP: {settings.google_cloud_project}")
print(f"Neo4j URI: {settings.neo4j_uri}")
print(f"Chunk size: {settings.chunk_size}")

Proyecto GCP: holmesgraphrag
Neo4j URI: bolt://localhost:7687
Chunk size: 1500


## 1. Inicializar Neo4j y crear esquema

In [2]:
neo4j = Neo4jManager()
neo4j.setup_database()  # Crea constraints + índices vectoriales + fulltext
print("Base de datos inicializada")
print(f"Stats actuales: {neo4j.get_stats()}")

Base de datos inicializada
Stats actuales: {}


## 2. Descargar y procesar textos de Gutenberg

In [3]:
processor = TextProcessor(neo4j_manager=neo4j)

#Solo los 10 relatos de desarrollo
story_chunks = processor.process_phase1()

print(f"\nRelatos procesados: {len(story_chunks)}")
for title, chunks in story_chunks.items():
    print(f"  - {title}: {len(chunks)} chunks")

Generando embeddings: 100%|██████████| 27/27 [00:00<00:00, 57.13it/s]   
                                                                             


Relatos procesados: 6
  - A Scandal In Bohemia: 32 chunks
  - The Red-Headed League: 34 chunks
  - The Adventure Of The Speckled Band: 36 chunks
  - The Adventure Of The Dancing Men: 35 chunks
  - Silver Blaze: 36 chunks
  - The Final Problem: 27 chunks


## 3. Extracción de entidades y relaciones

Extracción multipaso con sliding context:
- **Paso 1**: Extracción de entidades (Characters, Locations, Crimes, Objects, Deductions, Scenes, Events)
- **Paso 2**: Extracción de relaciones entre las entidades encontradas
- **Entity Resolution**: Normalización + embeddings + LLM para desambiguar duplicados

In [4]:
'''
import logging
logging.getLogger('graphrag.ingestion.entity_extractor').setLevel(logging.DEBUG)

extractor = EntityExtractor()

all_results = {}
story_title = "The Adventure Of The Speckled Band"
chunks = story_chunks[story_title]

# Solo 3 chunks para debug rápido
result = extractor.process_story_chunks(chunks, story_title)
all_results[story_title] = result

entities = result["entities"]
print(f"Personajes: {len(entities.get('characters', []))}")
for c in entities["characters"]:
  print(f"  [{c['name']}] aliases: {c.get('aliases', [])}")
'''

'\nimport logging\nlogging.getLogger(\'graphrag.ingestion.entity_extractor\').setLevel(logging.DEBUG)\n\nextractor = EntityExtractor()\n\nall_results = {}\nstory_title = "The Adventure Of The Speckled Band"\nchunks = story_chunks[story_title]\n\n# Solo 3 chunks para debug rápido\nresult = extractor.process_story_chunks(chunks, story_title)\nall_results[story_title] = result\n\nentities = result["entities"]\nprint(f"Personajes: {len(entities.get(\'characters\', []))}")\nfor c in entities["characters"]:\n  print(f"  [{c[\'name\']}] aliases: {c.get(\'aliases\', [])}")\n'

In [5]:
extractor = EntityExtractor()

# Para prueba — quitar el slice para el run completo
#TEST_STORIES = ["The Final Problem", "A Case Of Identity", "The Red-Headed League"]

all_results = {}
for story_title, chunks in story_chunks.items():
#    if story_title not in TEST_STORIES:
#        continue

    print(f"\n{'='*60}")
    print(f"Procesando: {story_title}")
    print(f"{'='*60}")

    result = extractor.process_story_chunks(chunks, story_title)
    all_results[story_title] = result

    # Resumen
    entities = result["entities"]
    n_chars = len(entities.get("characters", []))
    n_locs = len(entities.get("locations", []))
    n_crimes = len(entities.get("crimes", []))
    n_deductions = len(entities.get("deductions", []))
    print(f"  Personajes: {n_chars}, Ubicaciones: {n_locs}, Crímenes: {n_crimes}, Deducciones: {n_deductions}")
    print(f"  Relaciones: {len(result['relationships'])}")


Procesando: A Scandal In Bohemia


Generando embeddings: 100%|██████████| 77/77 [00:03<00:00, 22.28it/s]


  Personajes: 13, Ubicaciones: 11, Crímenes: 3, Deducciones: 50
  Relaciones: 232

Procesando: The Red-Headed League


Generando embeddings: 100%|██████████| 75/75 [00:03<00:00, 21.95it/s]


  Personajes: 15, Ubicaciones: 11, Crímenes: 2, Deducciones: 34
  Relaciones: 278

Procesando: The Adventure Of The Speckled Band


...

  Personajes: 13, Ubicaciones: 19, Crímenes: 2, Deducciones: 65
  Relaciones: 406

Procesando: Silver Blaze


Generando embeddings: 100%|██████████| 96/96 [00:03<00:00, 25.25it/s]


  Personajes: 20, Ubicaciones: 19, Crímenes: 2, Deducciones: 67
  Relaciones: 443

Procesando: The Final Problem


Extrayendo 'The Final Problem': 100%|██████████| 27/27 [14:33<00:00, 32.33s/it]
'The Final Problem': 1/27 chunks sin entidades (4%). Posibles fallos de API.
Generando embeddings: 100%|██████████| 77/77 [00:03<00:00, 22.22it/s]


  Personajes: 10, Ubicaciones: 8, Crímenes: 8, Deducciones: 30
  Relaciones: 197


In [6]:
import json
import os

# Guarda los resultados de extracción a disco por si el pipeline se interrumpe.
# Para recargar sin re-extraer: all_results = json.load(open("../output/extraction_results.json"))
os.makedirs("../output", exist_ok=True)
with open("../output/extraction_results.json", "w", encoding="utf-8") as f:
    json.dump(all_results, f, ensure_ascii=False, indent=2)

print(f"Checkpoint guardado: ../output/extraction_results.json ({len(all_results)} relatos)")

Checkpoint guardado: ../output/extraction_results.json (6 relatos)


In [7]:
# Resolución cross-story: unifica nombres canónicos entre relatos.
# Garantiza que Holmes y Watson tengan el mismo nombre canónico en Neo4j
# independientemente del relato de origen.
all_results = extractor.normalize_cross_story_entities(all_results)

# Guarda el checkpoint normalizado (sobreescribe el anterior)
with open("../output/extraction_results.json", "w", encoding="utf-8") as f:
    json.dump(all_results, f, ensure_ascii=False, indent=2)

print("Cross-story normalization completada.")
# Verificación rápida
for story, result in all_results.items():
    chars = result["entities"].get("characters", [])
    holmes = next((c["name"] for c in chars if "holmes" in c["name"].lower()), "--")
    watson = next((c["name"] for c in chars if "watson" in c["name"].lower()), "--")
    print(f"  {story[:40]:40s}  Holmes='{holmes}'  Watson='{watson}'")

Cross-story normalization completada.
  A Scandal In Bohemia                      Holmes='Sherlock Holmes'  Watson='Watson'
  The Red-Headed League                     Holmes='Sherlock Holmes'  Watson='Watson'
  The Adventure Of The Speckled Band        Holmes='Sherlock Holmes'  Watson='Watson'
  The Adventure Of The Dancing Men          Holmes='Sherlock Holmes'  Watson='Watson'
  Silver Blaze                              Holmes='Sherlock Holmes'  Watson='Watson'
  The Final Problem                         Holmes='Sherlock Holmes'  Watson='Watson'


## 4. Poblar el grafo en Neo4j

In [8]:
for story_title, result in all_results.items():
    print(f"Almacenando: {story_title}")

    # Almacenar entidades
    neo4j.store_entities(result["entities"], story_title)

    # Almacenar relaciones
    neo4j.store_relationships(result["relationships"], story_title=story_title)

    # Vincular chunks con las entidades que mencionan
    for chunk_info in result.get("chunk_entities", []):
        if chunk_info["chunk_id"] and chunk_info["entity_names"]:
            neo4j.link_chunk_to_entities(chunk_info["chunk_id"], chunk_info["entity_names"])

print("\nGrafo poblado exitosamente")

Almacenando: A Scandal In Bohemia
Almacenando: The Red-Headed League
Almacenando: The Adventure Of The Speckled Band
Almacenando: The Adventure Of The Dancing Men
Almacenando: Silver Blaze
Almacenando: The Final Problem

Grafo poblado exitosamente


## 5. Verificar el grafo

In [9]:
stats = neo4j.get_stats()
print("Estadísticas del grafo:")
for label, count in stats.items():
    print(f"  {label}: {count}")

Estadísticas del grafo:
  Event: 412
  Deduction: 323
  Object: 269
  Chunk: 200
  Scene: 185
  Location: 85
  Character: 73
  Crime: 19
  Story: 6


In [10]:
# Ver personajes más conectados
top_characters = neo4j.execute_query("""
MATCH (c:Character)-[r]-()
RETURN c.name AS name, count(DISTINCT r) AS connections
ORDER BY connections DESC
LIMIT 10
""")

print("\nPersonajes más conectados:")
for char in top_characters:
    print(f"  {char['name']}: {char['connections']} conexiones")


Personajes más conectados:
  Sherlock Holmes: 495 conexiones
  Watson: 216 conexiones
  Hilton Cubitt: 63 conexiones
  Colonel Ross: 55 conexiones
  John Straker: 49 conexiones
  Jabez Wilson: 48 conexiones
  Dr. Grimesby Roylott: 34 conexiones
  Irene Adler: 29 conexiones
  Elsie Patrick: 27 conexiones
  Helen Stoner: 27 conexiones


In [11]:
# Ver relatos y sus entidades
stories = neo4j.execute_query("""
MATCH (s:Story)
OPTIONAL MATCH (c:Character)-[:APPEARS_IN]->(s)
RETURN s.title AS story, s.collection AS collection, count(c) AS characters
ORDER BY characters DESC
""")

print("\nRelatos cargados:")
for s in stories:
    print(f"  {s['story']} ({s['collection']}): {s['characters']} personajes")


Relatos cargados:
  Silver Blaze (The Memoirs of Sherlock Holmes): 19 personajes
  The Red-Headed League (The Adventures of Sherlock Holmes): 15 personajes
  The Adventure Of The Speckled Band (The Adventures of Sherlock Holmes): 14 personajes
  A Scandal In Bohemia (The Adventures of Sherlock Holmes): 13 personajes
  The Adventure Of The Dancing Men (The Return of Sherlock Holmes): 13 personajes
  The Final Problem (The Memoirs of Sherlock Holmes): 10 personajes


In [12]:
# Ver cadenas de deducción
deductions = neo4j.execute_query("""
MATCH (d:Deduction)-[:LEADS_TO]->(d2:Deduction)
RETURN d.observation AS from_obs, d2.observation AS to_obs
LIMIT 5
""")

if deductions:
    print("\nCadenas de deducción encontradas:")
    for d in deductions:
        print(f"  {d['from_obs'][:60]}... → {d['to_obs'][:60]}...")
else:
    print("\nNo se encontraron cadenas de deducción (LEADS_TO)")


Cadenas de deducción encontradas:
  The narrator has frequently seen the steps leading from the ... → Holmes has both seen and observed the steps leading from the...
  The paper upon which the note was written.... → The peculiar quality of the paper....
  The peculiar quality of the paper.... → A large 'E' with a small 'g,' a 'P,' and a large 'G' with a ...
  A large 'E' with a small 'g,' a 'P,' and a large 'G' with a ... → The 'Eg' from the watermark, combined with the German origin...
  The paper was made in Bohemia.... → The peculiar construction of the sentence—'This account of y...


## 6. Cleanup (opcional)

In [13]:
# Descomentar para limpiar la base de datos completa
#neo4j.clear_database()
#print("Base de datos limpiada")

neo4j.close()
print("Conexión cerrada")

Conexión cerrada
